# Core format model examples

This notebook exercises the complete public **model interface**: every exported model type, both validated construction entry points, serialization, JSON Schema, metadata, normalization, copying, immutability, and validation errors.

Launch from the repository root with:

```sh
uv run --with jupyterlab jupyter lab examples/core_format_examples.ipynb
```

> The notebook imports directly from `src/`, so an editable package install is not required. Run cells from top to bottom.

In [1]:
import sys
from pathlib import Path

repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").exists()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Open this notebook from inside the apertus-common repository")
src = str(repo_root / "src")
if src not in sys.path:
    sys.path.insert(0, src)
print(f"Using repository: {repo_root}")

Using repository: /Users/rkre/swissai_repos/apertus-common


In [2]:
from collections.abc import Callable
from pprint import pprint

from jsonschema import Draft202012Validator
from pydantic import BaseModel, ValidationError

import apertus_common.model as model_api
from apertus_common import (
    ApertusFormatError,
    Assistant,
    Attachment,
    AudioPart,
    Conversation,
    Document,
    Effort,
    Event,
    GenericInput,
    GenericOutput,
    Harness,
    HeaderError,
    ImagePart,
    InputMessage,
    OutputMessage,
    Retrieval,
    SystemPrompt,
    TextPart,
    Think,
    ToolCall,
    ToolResult,
    ToolSpec,
    TypeInfo,
    User,
    VerifiableAnswer,
    Wait,
    delivery_order,
    is_canonical_id,
    type_info,
)

assert issubclass(HeaderError, ApertusFormatError)
print("Public model exports:")
print(", ".join(model_api.__all__))

Public model exports:
Assistant, Attachment, AudioPart, Conversation, Document, Effort, Event, GenericInput, GenericOutput, Harness, ImagePart, InputMessage, OutputMessage, Retrieval, SystemPrompt, TextPart, Think, ToolCall, ToolResult, ToolSpec, TypeInfo, User, VerifiableAnswer, Wait, delivery_order, is_canonical_id, type_info


## 1. Class hierarchy and fields

All concrete values are Pydantic `BaseModel` subclasses. `InputMessage` and `OutputMessage` are public base classes; applications normally instantiate their concrete subclasses.

In [3]:
input_classes: list[type[InputMessage]] = [
    Harness,
    ToolResult,
    Retrieval,
    Event,
    Attachment,
    User,
    Document,
    GenericInput,
]
output_classes: list[type[OutputMessage]] = [
    Assistant,
    Think,
    ToolCall,
    VerifiableAnswer,
    GenericOutput,
]
standalone_classes: list[type[BaseModel]] = [
    Conversation,
    Wait,
    SystemPrompt,
    ToolSpec,
    TextPart,
    ImagePart,
    AudioPart,
]

assert all(issubclass(cls, InputMessage) for cls in input_classes)
assert all(issubclass(cls, OutputMessage) for cls in output_classes)
assert all(issubclass(cls, BaseModel) for cls in standalone_classes)

model_classes: list[type[BaseModel]] = [
    *input_classes,
    *output_classes,
    *standalone_classes,
]
for cls in model_classes:
    fields = ", ".join(cls.model_fields)
    print(f"{cls.__name__:18} -> {fields}")

Harness            -> extras, direction, type, payload
ToolResult         -> extras, direction, type, id, payload
Retrieval          -> extras, direction, type, src, payload
Event              -> extras, direction, type, payload
Attachment         -> extras, direction, type, name, mime, payload
User               -> extras, direction, type, payload
Document           -> extras, direction, type, payload
GenericInput       -> extras, direction, type, payload
Assistant          -> extras, direction, type, payload
Think              -> extras, direction, type, payload
ToolCall           -> extras, direction, type, id, name, args
VerifiableAnswer   -> extras, direction, type, payload
GenericOutput      -> extras, direction, type, payload
Conversation       -> system, items
Wait               -> type
SystemPrompt       -> identity, behavior, effort, tools, environment, extra_tags
ToolSpec           -> name, json_schema, policy
TextPart           -> part, text
ImagePart          -> part, ref,

## 2. Primitives and multimodal parts

A string payload for `User` or `Attachment` is normalized to an immutable one-element `TextPart` sequence. Image geometry declares both dimensions or neither.

In [4]:
text_part = TextPart(text="Describe this media")
image_part = ImagePart(ref="asset://image-1", height=512, width=768)
audio_part = AudioPart(ref="asset://audio-1")
user_from_text = User.model_validate({"payload": "Hello"})

assert isinstance(user_from_text.payload[0], TextPart)
assert Effort("medium") is Effort.MEDIUM
pprint([text_part, image_part, audio_part, user_from_text])

[TextPart(part='text', text='Describe this media'),
 ImagePart(part='image', ref='asset://image-1', height=512, width=768),
 AudioPart(part='audio', ref='asset://audio-1'),
 User(extras={}, direction='in', type='user', payload=[TextPart(part='text', text='Hello')])]


## 3. System prompt and tools

In [5]:
weather_tool = ToolSpec(
    name="get_weather",
    json_schema={
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
    policy="Use when current weather is needed.",
)
system = SystemPrompt(
    identity="You are Aria.",
    behavior="Be concise and cite tool results.",
    effort=Effort.MEDIUM,
    tools=[weather_tool],
    environment="production",
    extra_tags={"locale": "en-CH"},
)
pprint(system.model_dump(mode="python"))

{'behavior': 'Be concise and cite tool results.',
 'effort': <Effort.MEDIUM: 'medium'>,
 'environment': 'production',
 'extra_tags': {'locale': 'en-CH'},
 'identity': 'You are Aria.',
 'tools': [{'json_schema': {'properties': {'city': {'type': 'string'}},
                            'required': ['city'],
                            'type': 'object'},
            'name': 'get_weather',
            'policy': 'Use when current weather is needed.'}]}


## 4. Every input, output, and stop item

In [6]:
inputs = [
    Harness(payload="The UI locale is en-CH.", extras={"source": "ui"}),
    ToolResult(id="get_weather:0", payload='{"tempC":19,"condition":"rain"}'),
    Retrieval(src="weather_docs", payload="Rain jackets block wind and water."),
    Event(payload="The user enabled location access."),
    Attachment(name="street.jpg", mime="image/jpeg", payload=[image_part]),
    User(payload=[text_part, image_part, audio_part]),
    Document(payload="Raw pretraining document content."),
    GenericInput(type="memory", payload="Deployment-defined input."),
]

outputs = [
    Assistant(payload="Bring a waterproof jacket."),
    Think(payload="I should use the current weather tool."),
    ToolCall(id="get_weather:0", name="get_weather", args={"city": "Zurich"}),
    VerifiableAnswer(payload="19"),
    GenericOutput(type="debug_trace", payload="Deployment-defined output."),
]
wait = Wait()

for item in [*inputs, *outputs, wait]:
    print(f"{type(item).__name__:18} {item.model_dump(mode='json')}")

Harness            {'extras': {'source': 'ui'}, 'direction': 'in', 'type': 'harness', 'payload': 'The UI locale is en-CH.'}
ToolResult         {'extras': {}, 'direction': 'in', 'type': 'tool_result', 'id': 'get_weather:0', 'payload': '{"tempC":19,"condition":"rain"}'}
Retrieval          {'extras': {}, 'direction': 'in', 'type': 'retrieval', 'src': 'weather_docs', 'payload': 'Rain jackets block wind and water.'}
Event              {'extras': {}, 'direction': 'in', 'type': 'event', 'payload': 'The user enabled location access.'}
Attachment         {'extras': {}, 'direction': 'in', 'type': 'attachment', 'name': 'street.jpg', 'mime': 'image/jpeg', 'payload': [{'part': 'image', 'ref': 'asset://image-1', 'height': 512, 'width': 768}]}
User               {'extras': {}, 'direction': 'in', 'type': 'user', 'payload': [{'part': 'text', 'text': 'Describe this media'}, {'part': 'image', 'ref': 'asset://image-1', 'height': 512, 'width': 768}, {'part': 'audio', 'ref': 'asset://audio-1'}]}
Document   

## 5. Construction entry points

Use direct constructors when types are already known. Use `Conversation.model_validate(...)` for untyped Python data and `Conversation.model_validate_json(...)` for JSON; the callable discriminator restores each concrete item class.

In [7]:
conversation = Conversation(
    system=system,
    items=[
        Harness(payload="Locale: en-CH"),
        User(payload=[TextPart(text="What should I wear in Zurich?")]),
        Think(payload="Need current weather."),
        ToolCall(id="get_weather:0", name="get_weather", args={"city": "Zurich"}),
        Wait(),
        ToolResult(id="get_weather:0", payload='{"tempC":19,"condition":"rain"}'),
        Assistant(payload="Wear a waterproof jacket."),
        Wait(),
    ],
)

from_python = Conversation.model_validate(conversation.model_dump(mode="python"))
from_json = Conversation.model_validate_json(conversation.model_dump_json())
assert from_python == from_json == conversation
assert [type(item) for item in from_json.items] == [
    type(item) for item in conversation.items
]
print([type(item).__name__ for item in from_json.items])

['Harness', 'User', 'Think', 'ToolCall', 'Wait', 'ToolResult', 'Assistant', 'Wait']


In [8]:
# Unknown types are preserved through their direction-specific generic class.
unknown = Conversation.model_validate(
    {"items": [{"direction": "in", "type": "sensor", "payload": "18.7C"}]}
)
assert isinstance(unknown.items[0], GenericInput)
unknown.items[0]

GenericInput(extras={}, direction='in', type='sensor', payload='18.7C')

## 6. Dumping and JSON Schema

`model_dump(mode="python")` returns Python-oriented values, `model_dump(mode="json")` returns JSON-ready data, and `model_dump_json()` returns encoded JSON text.

In [9]:
python_data = conversation.model_dump(mode="python")
json_data = conversation.model_dump(mode="json")
json_text = conversation.model_dump_json(indent=2)

print(json_text[:900] + "\n...")
print(f"Python root type: {type(python_data).__name__}")
print(f"JSON text size: {len(json_text)} bytes")

{
  "system": {
    "identity": "You are Aria.",
    "behavior": "Be concise and cite tool results.",
    "effort": "medium",
    "tools": [
      {
        "name": "get_weather",
        "json_schema": {
          "type": "object",
          "properties": {
            "city": {
              "type": "string"
            }
          },
          "required": [
            "city"
          ]
        },
        "policy": "Use when current weather is needed."
      }
    ],
    "environment": "production",
    "extra_tags": {
      "locale": "en-CH"
    }
  },
  "items": [
    {
      "extras": {},
      "direction": "in",
      "type": "harness",
      "payload": "Locale: en-CH"
    },
    {
      "extras": {},
      "direction": "in",
      "type": "user",
      "payload": [
        {
          "part": "text",
          "text": "What should I wear in Zurich?"
        }
      ]
    },
    
...
Python root type: dict
JSON text size: 1588 bytes


In [10]:
schema = Conversation.model_json_schema()
Draft202012Validator.check_schema(schema)
Draft202012Validator(schema).validate(json_data)  # pyright: ignore[reportUnknownMemberType]
print(f"Schema title: {schema['title']}")
print(f"Definitions: {len(schema.get('$defs', {}))}")
print("This conversation dump is valid under the exported Draft 2020-12 schema.")

Schema title: Conversation
Definitions: 28
This conversation dump is valid under the exported Draft 2020-12 schema.


### JSON lines for datasets

`model_dump_json()` on one line per conversation is the dataset convention:
each line is self-contained JSON, and `Conversation.model_validate_json`
restores the exact classes per line. `to_json`/`from_json` are the same two
calls under harmony-style names, with ingest flags (`normalize`, `check`,
`strict`); see the dataset notebook.

In [11]:
dataset = [
    conversation,
    Conversation(  # e.g. a pretraining sample: no system prompt
        items=[Document(payload="Once upon a time..."), Think(payload="Fiction.")]
    ),
]
jsonl_lines = [c.model_dump_json() for c in dataset]
print(jsonl_lines[1])

restored = [Conversation.model_validate_json(line) for line in jsonl_lines]
assert restored == dataset
print(f"\n{len(jsonl_lines)} conversations round-tripped as JSONL lines")

{"system":null,"items":[{"extras":{},"direction":"in","type":"document","payload":"Once upon a time..."},{"extras":{},"direction":"out","type":"think","payload":"Fiction."}]}

2 conversations round-tripped as JSONL lines


## 7. Type metadata and canonical normalization

Order and authority are class metadata, not serialized fields. `normalized()` stably sorts each consecutive input run; outputs and waits form boundaries and never move.

In [12]:
for message_type in [
    "harness",
    "tool_result",
    "retrieval",
    "event",
    "attachment",
    "user",
    "custom",
]:
    info: TypeInfo = type_info("in", message_type)
    print(f"{info.type:12} order={info.order!s:4} authority={info.authority}")

print("Delivery order:", {type(item).__name__: delivery_order(item) for item in inputs})
print(
    "Canonical ID checks:",
    {value: is_canonical_id(value) for value in ["tool:0", "call_123", "bad id"]},
)

harness      order=1    authority=2
tool_result  order=2    authority=4
retrieval    order=3    authority=4
event        order=3    authority=4
attachment   order=4    authority=4
user         order=5    authority=3
custom       order=3    authority=4
Delivery order: {'Harness': 1, 'ToolResult': 2, 'Retrieval': 3, 'Event': 3, 'Attachment': 4, 'User': 5, 'Document': 3, 'GenericInput': 3}
Canonical ID checks: {'tool:0': True, 'call_123': False, 'bad id': False}


In [13]:
messy = Conversation(
    items=[
        User(payload=[TextPart(text="last input")]),
        Event(payload="middle input"),
        Harness(payload="first input"),
        Assistant(payload="output boundary"),
        User(payload=[TextPart(text="second run")]),
        Harness(payload="start second run"),
    ]
)
normalized = messy.normalized()
print("Before:", [item.type for item in messy.items])
print("After: ", [item.type for item in normalized.items])
assert normalized.normalized() == normalized
assert messy.items[0].type == "user"  # original was not mutated

Before: ['user', 'event', 'harness', 'assistant', 'user', 'harness']
After:  ['harness', 'event', 'user', 'assistant', 'harness', 'user']


## 8. Copying, immutability, and validated changes

Use `model_copy(deep=True)` for an equivalent isolated copy. To change values, dump, modify ordinary data, and pass it through `model_validate` again. Avoid `model_construct()` and untrusted `model_copy(update=...)`: those Pydantic escape hatches intentionally bypass validation.

In [14]:
copied = conversation.model_copy(deep=True)
assert copied == conversation and copied is not conversation

changed_data = conversation.model_dump(mode="python")
changed_data["system"] = None
changed = Conversation.model_validate(changed_data)
assert changed.system is None and conversation.system is not None

try:
    conversation.system = None  # type: ignore[misc]
except ValidationError as error:
    print("Frozen field assignment rejected:", error.errors()[0]["type"])

try:
    system.extra_tags["new"] = "value"  # type: ignore[index]
except TypeError as error:
    print("Nested mapping mutation rejected:", error)

Frozen field assignment rejected: frozen_instance
Nested mapping mutation rejected: 'FrozenDict' object does not support item assignment


## 9. Validation errors

Model validators raise specific value errors such as `HeaderError`; Pydantic collects and exposes them as `ValidationError` with structured locations and messages.

In [15]:
invalid_examples: dict[str, Callable[[], object]] = {
    "bad header key": lambda: Harness(payload="x", extras={"bad key": "value"}),
    "invalid call id": lambda: ToolCall(id="bad id", name="tool", args={}),
    "non-finite JSON": lambda: ToolCall(
        id="tool:0", name="tool", args={"x": float("inf")}
    ),
    "partial geometry": lambda: ImagePart(height=10),
    "known generic type": lambda: GenericInput(type="user", payload="x"),
}

for label, construct in invalid_examples.items():
    try:
        construct()
    except ValidationError as error:
        detail = error.errors()[0]
        print(f"{label:20} -> {detail['loc']}: {detail['msg']}")

bad header key       -> ('extras',): Value error, header key 'bad key' violates the key grammar [A-Za-z0-9_]+
invalid call id      -> ('id',): Value error, tool-call id 'bad id' does not match the permissive id shape [A-Za-z0-9_:.-]{1,64}
non-finite JSON      -> ('args', 'bool'): Input should be a valid boolean
partial geometry     -> (): Value error, image geometry declares both height and width, or neither
known generic type   -> ('type',): Value error, 'user' is a known input type; use its dedicated class


## 10. Rendering to the wire format

`render_str()` serializes a conversation to the Apertus string form.
The mechanism behind it is the **wire IR**, an intermediate
representation between the typed model and the wire: the renderer
walks the model once into a flat sequence of atoms
(`Control` framing tokens, `Text` runs with their trust origin,
`Media` placeholders), and each serialization is a pure projection
of that same sequence:

- the **string projection** (implemented) maps control atoms to their
  surface glyphs (`<|in|>`, `<|hdr|>`, ...) and text verbatim;
- the planned **token projection** will map control atoms to their
  registered single token ids and encode payload text with
  special-token recognition disabled.

In the string form, payload text is verbatim and unescaped, so a
payload that spells a control glyph is ambiguous there. The planned token
path preserves unforgeability by inserting controls structurally and
encoding payload text with special-token recognition disabled.

The experimental `StreamParser` is a strict reverse path for one
contiguous model-output segment only. It is not a full transcript parser,
and its API may change while token and serving integration develops. Full
transcript `parse_str` remains optional. Semantic reporting remains the
separate model-only `check(conversation)`. Rendering never reorders or
validates: use `normalized()` and `check()` first.

In [16]:
from apertus_common import render_str

print(render_str(conversation))

<|sys|>
<identity>You are Aria.</identity>
<behavior>Be concise and cite tool results.</behavior>
<effort>medium</effort>
<tools>
  <tool name="get_weather">
    <schema>{"type":"object","properties":{"city":{"type":"string"}},"required":["city"]}</schema>
    <policy>Use when current weather is needed.</policy>
  </tool>
</tools>
<locale>en-CH</locale>
<environment>production</environment>
<|/sys|>

<|in|> type=harness <|hdr|> Locale: en-CH <|/in|>

<|in|> type=user <|hdr|> What should I wear in Zurich? <|/in|>

<|out|> type=think <|hdr|> Need current weather. <|/out|>

<|out|> type=tool_call id=get_weather:0 <|hdr|> {"name":"get_weather","args":{"city":"Zurich"}} <|/out|>
<|wait|>

<|in|> type=tool_result id=get_weather:0 <|hdr|> {"tempC":19,"condition":"rain"} <|/in|>

<|out|> type=assistant <|hdr|> Wear a waterproof jacket. <|/out|>
<|wait|>


## Interface cheat sheet

| Goal | Interface |
|---|---|
| Construct a known type | `User(...)`, `ToolCall(...)`, `SystemPrompt(...)`, etc. |
| Validate untyped Python data | `Conversation.model_validate(data)` |
| Validate JSON text | `Conversation.model_validate_json(text)` |
| Dump Python / JSON-ready data | `model.model_dump(mode="python" / "json")` |
| Dump JSON text | `model.model_dump_json()` |
| Dataset file (JSON lines) | one `to_json(conversation)` per line, `from_json(line)` per line |
| Export JSON Schema | `Conversation.model_json_schema()` |
| Inspect declared fields | `ModelClass.model_fields` |
| Make an equivalent copy | `model.model_copy(deep=True)` |
| Canonicalize input runs | `conversation.normalized()` |
| Render to the string format | `render_str(conversation)` |
| Look up order / authority | `type_info(direction, type)` |
| Resolve an input's delivery order | `delivery_order(message)` |
| Check canonical call IDs | `is_canonical_id(value)` |

In [17]:
# Scratch space: edit and rerun this cell.
my_conversation = Conversation(
    system=SystemPrompt(effort=Effort.LOW),
    items=[
        User(payload=[TextPart(text="Hello, Apertus!")]),
        Assistant(payload="Hello!"),
        Wait(),
    ],
)
print(my_conversation.model_dump_json(indent=2))

{
  "system": {
    "identity": null,
    "behavior": null,
    "effort": "low",
    "tools": [],
    "environment": null,
    "extra_tags": {}
  },
  "items": [
    {
      "extras": {},
      "direction": "in",
      "type": "user",
      "payload": [
        {
          "part": "text",
          "text": "Hello, Apertus!"
        }
      ]
    },
    {
      "extras": {},
      "direction": "out",
      "type": "assistant",
      "payload": "Hello!"
    },
    {
      "type": "wait"
    }
  ]
}
